# Robust false-constraint arbitration on a nonneg tensor *tree* — the batched path

The single-variable [robust_false_constraint](robust_false_constraint.ipynb) notebook and
its 20-variable [chain port](robust_false_constraint_large.ipynb) are `kind="born"`. This
one exercises the **`kind="nonneg"` tensor tree** through its batched constraint machinery
(`SteinerMarginals` + `BPPlan`): the path built for *many heterogeneous cross-variable
constraints* whose per-constraint compile graph would otherwise wall at scale.

**30 continuous variables** on `[0,100]` sit at the leaves of a balanced binary tree
(dim-1 latent junctions, max degree 3). Ground truth is a set of independent discretised
gaussians, so every *trusted* cross-variable target is mutually consistent (a product
distribution the tree can represent) and a good fit can drive all trusted residuals to ~0.

Two phases:

| phase | constraints | what it checks |
|---|---|---|
| **A - ordinary** | 50 trusted (KL singles, indicator-product pairs & triples, rare-event conditionals) + 3 flatly-false, all *hard* | the batched nonneg path fits end-to-end under `optimize`; false hard terms leave residual and drag the variables they share |
| **B - robust** | same 50 trusted, but the 3 suspects enter as **on/off trust latents** (`onoff_expectation`) | self-discard: the false latents flee to *broken* (`q_active -> 0`), their variables recover the truth, the loss lands on the `-log p_broken` floor |

> **Set up but not executed** - run top to bottom. Phase A uses the batched
> `BPPlan` sweep (uniform leaf dims); Phase B's dim-2 trust latents make leaf dims
> mixed, so it steps aside to the exact per-node fallback sweep (same numbers, ~4x
> slower at this size). The `klw` knob in Phase B is the one calibration lesson -
> see the note there.

## 0. Setup - the tree, the ground truth, and the trusted constraint builder

In [ ]:
import os
os.environ.setdefault('JAX_PLATFORMS', 'cpu')
import numpy as np, jax, jax.numpy as jnp
from calibrated_response.tn import TensorTree, ContinuousVar, latent_var, losses as L
from calibrated_response.tn.steiner import BPPlan

NV, NB, R = 30, 12, 6              # data variables, bins per variable, bond dim
LO, HI = 0.0, 100.0
ind = lambda t: (lambda x: (np.asarray(x) > t).astype(np.float64))   # indicator 1{x>t}


def latent_tree(n_leaves):
    '''Balanced binary tree over n_leaves observables: dim-1 latent junctions, max degree 3.'''
    frontier = list(range(n_leaves)); nxt = n_leaves; edges = []
    while len(frontier) > 1:
        new = []
        for k in range(0, len(frontier), 2):
            if k + 1 < len(frontier):
                lat = nxt; nxt += 1
                edges += [(lat, frontier[k]), (lat, frontier[k + 1])]; new.append(lat)
            else:
                new.append(frontier[k])
        frontier = new
    return edges, nxt


def ground_truth(model):
    '''Independent discretised gaussians -> every trusted cross-variable target is mutually
    consistent (a product distribution the tree can represent). Returns bin centres, the
    per-variable pmfs, their means, the {X>50} masks, and P(X>50).'''
    r = np.random.default_rng(0)
    cen = model.disc.bin_centers(0)
    means = r.uniform(30, 70, NV); sds = r.uniform(12, 22, NV)
    pmf = np.stack([(lambda q: q / q.sum())(np.exp(-0.5 * ((cen - means[i]) / sds[i]) ** 2))
                    for i in range(NV)])
    true_mean = (cen * pmf).sum(1)
    mask = {i: np.asarray(model.threshold_mask(i, 50.)) for i in range(NV)}
    true_pgt = np.array([float((mask[i] * pmf[i]).sum()) for i in range(NV)])
    return dict(cen=cen, pmf=pmf, true_mean=true_mean, mask=mask, true_pgt=true_pgt)


def trusted_constraints(model, gt, klw):
    '''50 mutually-consistent cross-variable constraints: one KL per variable (pins each
    marginal, priced `klw`), 12 indicator-product pairs and 4 triples spanning different
    branches, and 4 rare-event conditionals. Consistent under the independent ground truth.'''
    r = np.random.default_rng(1)
    mask, pmf, tp = gt['mask'], gt['pmf'], gt['true_pgt']
    cons = [('kl', i, jnp.asarray(pmf[i], jnp.float32), klw) for i in range(NV)]
    pair_ij = [(int(i), int(j)) for i, j in
               ((r.integers(NV), r.integers(NV)) for _ in range(12)) if i != j]
    for i, j in pair_ij:
        cons.append(('expect', ((i, j), np.outer(mask[i], mask[j]).astype(np.float32)),
                     float(tp[i] * tp[j]), 4.0))
    trip = [tuple(sorted(r.choice(NV, 3, replace=False).tolist())) for _ in range(4)]
    for a, b, c in trip:
        cons.append(('expect', ((a, b, c),
            np.einsum('a,b,c->abc', mask[a], mask[b], mask[c]).astype(np.float32)),
            float(tp[a] * tp[b] * tp[c]), 4.0))
    cond_ab = [(int(a), int(b)) for a, b in
               ((r.integers(NV), r.integers(NV)) for _ in range(4)) if a != b]
    for a, b in cond_ab:
        cons.append(('cond', {a: model.threshold_mask(a, 50.)},
                     {b: model.threshold_mask(b, 50.)}, float(tp[a])))
    return cons, pair_ij, trip, cond_ab


def resid(model, p, cst):
    '''Interpretable per-constraint residual: KL in nats, expect/cond in prob/value units.'''
    if cst[0] == 'kl':     return float(model.marginal_kl(p, cst[1], cst[2]))
    if cst[0] == 'expect': return float(abs(model.expectation(p, cst[1]) - cst[2]))
    return float(abs(model.cond_prob(p, cst[1], cst[2]) - cst[3]))

## 1. Phase A - the batched nonneg path, all 53 constraints *hard*

The 30 data leaves give a uniform-dim tree, so the batched `BPPlan` sweep is active
(`ok=True`): one up + one down level-synchronous pass, `O(depth)` einsums rather than one
per node. `constraint_loss` scores all 53 constraints off that single sweep, and its
numbers match the per-constraint reference exactly. The 3 false constraints are ordinary
hard squared-error terms - no notion of trust - so the fit cannot satisfy them and, being
hard, they drag the variables they touch.

In [ ]:
edges, ntot = latent_tree(NV); n_lat = ntot - NV
mA = TensorTree([ContinuousVar(f'X{i}', LO, HI, NB) for i in range(NV)]
                + [ContinuousVar(f'h{k}', 0., 1., 1) for k in range(n_lat)],
                edges=edges, bond_dim=R, kind='nonneg')
gt = ground_truth(mA)
trusted, pair_ij, trip, cond_ab = trusted_constraints(mA, gt, klw=1.0)

# 3 flatly-false constraints, each contradicting the truth on its own variable
f0, f1 = 22, 21; (f3, f4) = cond_ab[0]
false = [
    ('expect', {f0: jnp.asarray(gt['cen'], jnp.float32)}, float(100 - gt['true_mean'][f0]), 4e-3),
    ('expect', {f1: jnp.asarray(gt['mask'][f1], jnp.float32)},
     float(0.9 if gt['true_pgt'][f1] < 0.5 else 0.05), 4.0),
    ('cond', {f3: mA.threshold_mask(f3, 50.)}, {f4: mA.threshold_mask(f4, 50.)},
     float(0.9 if gt['true_pgt'][f3] < 0.5 else 0.05)),
]
cons = trusted + false

bp = BPPlan(mA)
print(f'tree: {ntot} nodes ({NV} data leaves + {n_lat} latents), NB={NB} r={R}')
print(f'BPPlan.ok={bp.ok}  batched sweep {len(bp.up_levels)}+{len(bp.down_levels)} ops vs {ntot} per-node')

# the batched provider gives identical numbers to the per-constraint reference
p0 = mA.init_params(seed=1)
d = abs(float(mA.constraint_loss(cons)(p0))
        - float(mA._cached_constraint_scores(mA._cores(p0), cons)))
print(f'batched vs per-constraint reference @init: |diff|={d:.2e}')

p, hist = mA.optimize(mA.constraint_loss(cons), backend='adam', steps=1500, lr=3e-2, seed=1)
print(f'fit: 1500 adam steps   loss {float(hist[0]):.3f} -> {float(hist[-1]):.4f}')

tr = np.array([resid(mA, p, c) for c in trusted])
print(f'trusted residuals ({len(trusted)}): mean={tr.mean():.4f}  max={tr.max():.4f}')
for c, name in zip(false, ['FALSE-mean', 'FALSE-prob', 'FALSE-cond']):
    print(f'  {name}: residual={resid(mA, p, c):.3f}')

The trusted residuals collapse to ~0 (the consistent cross-variable constraints are all
met) while the three false residuals stay large - and the *max* trusted residual sits on a
variable a false constraint shares, dragged off by the hard lie. That silent corruption,
with nothing flagging which constraint caused it, is exactly what the robust phase fixes.

## 2. Phase B - robustness: flag the 3 suspects as on/off trust latents

Same 30 variables and same 50 trusted constraints, but the 3 suspects now enter as
`onoff_expectation`: a 2-state latent (state 0 = *broken* / asserts nothing, state 1 =
*active*) whose penalty compares the **marginal** `E[f(X)]` against the target, gated by
the latent's own marginal `q_active`, plus `KL(q || [p_broken, 1-p_broken])`. Convicting a
constraint (`q_active -> 0`) switches its pull off for a flat price of `-log p_broken ~ 3`
nats. The three trust latents are just extra leaves on the tree - being dim-2 they make the
leaf dims non-uniform, so `BPPlan.ok=False` and the loss uses the exact per-node fallback
sweep (identical numbers, slower).

**The one calibration knob - `klw`.** Arbitration is only coherent if the trusted
constraints are priced in a currency that *dominates* the conviction floor. With `klw=1.0`
(the Phase A weight), a suspect can satisfy its lie by dragging its variable's marginal for
less than the 3-nat conviction price, so it never self-discards. Pricing the trusted KLs at
`klw=20` makes violating a trusted marginal cost more than convicting - and the suspects
convict instead. (This mirrors the born notebook's section 4 coherent-pricing rule: trusted
constraints are the same statement with a tiny `p_broken` / small `sd`.)

In [ ]:
edges, ntot = latent_tree(NV + 3); n_lat = ntot - (NV + 3)   # 30 data + 3 trust latents
mB = TensorTree([ContinuousVar(f'X{i}', LO, HI, NB) for i in range(NV)]
                + [latent_var(f't{k}', 0., 1., 2) for k in range(3)]
                + [ContinuousVar(f'h{k}', 0., 1., 1) for k in range(n_lat)],
                edges=edges, bond_dim=R, kind='nonneg')
TRUST = [NV, NV + 1, NV + 2]
gt = ground_truth(mB)
trusted, pair_ij, trip, cond_ab = trusted_constraints(mB, gt, klw=20.0)   # <- stiff trusted pricing

f0, f1 = 22, 21; (f3, f4) = cond_ab[0]; tp = gt['true_pgt']
t_mean = float(100 - gt['true_mean'][f0])
t_prob = 0.9 if tp[f1] < 0.5 else 0.05
t_cond = 0.9 if tp[f3] < 0.5 else 0.05
suspects = [
    (L.onoff_expectation(f0, TRUST[0], t_mean, 1.0, p_broken=0.05), 1.0),                 # false mean
    (L.onoff_expectation(f1, TRUST[1], t_prob, 0.02, f=ind(50.), p_broken=0.05), 1.0),    # false prob
    (L.onoff_expectation(f3, TRUST[2], t_cond, 0.02, f=ind(50.),
                         given={f4: mB.threshold_mask(f4, 50.)}, p_broken=0.05), 1.0),     # false cond
]
print(f'tree: {ntot} nodes (30 data + 3 trust + {n_lat} latents)')
print(f'BPPlan.ok={BPPlan(mB).ok}  (mixed leaf dims -> exact per-node fallback sweep)')
print('suspects (all planted false):')
print(f'  E[X{f0}]={t_mean:.1f} (true {gt["true_mean"][f0]:.1f})')
print(f'  P(X{f1}>50)={t_prob:.2f} (true {tp[f1]:.2f})')
print(f'  P(X{f3}>50|X{f4}>50)={t_cond:.2f} (true {tp[f3]:.2f})')

p, hist = mB.optimize(L.combined_loss(mB, trusted, suspects), backend='adam',
                      steps=2500, lr=3e-2, seed=1)
print(f'\nfit: 2500 adam steps   loss {float(hist[0]):.3f} -> {float(hist[-1]):.3f}'
      f'  (floor ~ 3*-log0.05 = {3 * -np.log(0.05):.2f} if all 3 convicted)')

trust = [float(np.asarray(mB.joint_marginal(p, s))[1]) for s in TRUST]
print(f'\ntrust q(active) on the 3 FALSE suspects: {np.round(trust, 3)}   (want ~0)')
rec_m = float(mB.expectation(p, {f0: jnp.asarray(gt['cen'], jnp.float32)}))
rec_p = float(mB.event_prob(p, {f1: mB.threshold_mask(f1, 50.)}))
rec_c = float(mB.cond_prob(p, {f3: mB.threshold_mask(f3, 50.)}, {f4: mB.threshold_mask(f4, 50.)}))
print('recovery on the suspects own variables (should track TRUTH, not the lie):')
print(f'  E[X{f0}]          = {rec_m:5.1f}   true {gt["true_mean"][f0]:5.1f}   lie {t_mean:5.1f}')
print(f'  P(X{f1}>50)        = {rec_p:.2f}    true {tp[f1]:.2f}    lie {t_prob:.2f}')
print(f'  P(X{f3}>50|X{f4}>50) = {rec_c:.2f}    true {tp[f3]:.2f}    lie {t_cond:.2f}')
tr = np.array([resid(mB, p, c) for c in trusted])
print(f'\ntrusted residuals ({len(trusted)}): mean={tr.mean():.4f}  max={tr.max():.4f}')

### Takeaway

All three false latents flee to *broken* (`q_active -> ~0`), their variables recover the
**truth rather than the lie**, the 50 trusted cross-variable constraints hold to ~1e-3, and
the final loss lands on `3*(-log 0.05) ~ 8.99` - the entire remaining loss is precisely the
prior cost of convicting the three false constraints, every fit term driven to zero. Robust
self-discard, reproduced on a nonneg tensor tree through the batched constraint path.

Two honest notes carry over. **Pricing coherence is load-bearing:** the `klw=1->20` change
is the difference between suspects self-discarding and suspects winning by dragging a
weakly-priced marginal - trusted constraints must dominate the conviction floor. **The
batched sweep needs uniform leaf dims:** the dim-2 trust latents drop Phase B onto the
exact per-node fallback (correct, ~4x slower here); batching the sweep under mixed leaf
dims is possible but out of scope for this demo.